# 02 — Universal 4-Class SegFormer-B2 (Adult Prêt + Kids Ethnic + Wedding Formals/Lehenga)
- Jointly trains **SegFormer-B2** on:
  1. Your **95 FYP-2 Gold Adult Prêt masks** (read-only)
  2. Your new **V3 Gold Kids + Wedding Formal/Lehenga COCO masks** (`gold_v3_masks/*.zip`)
- Full `pycocotools` RLE + Polygon rasterization in strict Pakistani layering order (`bottom -> kurta_kameez -> dupatta`).
- Reports **separate per-domain held-out IoU** (`Adult Prêt`, `Kids Ethnic`, `Wedding Formals`) vs Zero-Shot Baseline!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate pycocotools scikit-learn tqdm pillow numpy pandas opencv-python-headless


In [ ]:
import json
import zipfile
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from pycocotools import mask as mask_utils
from sklearn.model_selection import train_test_split

V3_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_V3_Kids_Formal_Extension')
FYP2_GOLD_DIR = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research/01_PK_VTON_DATASET/gold_4class_masks')
V3_GOLD_DIR = V3_ROOT / '01_KIDS_AND_FORMAL_DATA' / 'gold_v3_masks'
RUN_DIR = V3_ROOT / '02_SEGFORMER_V3_UNIVERSAL' / 'segformer_b2_v3_universal'
RUN_DIR.mkdir(parents=True, exist_ok=True)
PAPER_DIR = V3_ROOT / '05_SUPERVISOR_REPORT_ARTIFACTS'
PAPER_DIR.mkdir(parents=True, exist_ok=True)

LAYER_ORDER = [2, 1, 3]  # bottom -> kurta_kameez -> dupatta on top

def rasterize_coco_zip(zip_path: Path, out_dir: Path, default_domain: str):
    ext_dir = out_dir / f'extracted_{zip_path.stem}'
    mask_dir = out_dir / f'masks_{zip_path.stem}'
    ext_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(ext_dir)
    rows = []
    for cj in ext_dir.rglob('_annotations.coco.json'):
        data = json.loads(cj.read_text(encoding='utf-8'))
        cat_to_canon = {}
        for c in data.get('categories', []):
            cn = str(c['name']).strip().lower()
            if 'kurta' in cn or 'kameez' in cn or 'choli' in cn or 'shirt' in cn:
                cat_to_canon[c['id']] = 1
            elif 'bottom' in cn or 'shalwar' in cn or 'lehenga' in cn or 'trouser' in cn:
                cat_to_canon[c['id']] = 2
            elif 'dupatta' in cn or 'shawl' in cn or 'scarf' in cn:
                cat_to_canon[c['id']] = 3
        anns_by_img = {}
        for a in data.get('annotations', []):
            anns_by_img.setdefault(a['image_id'], []).append(a)
        for info in data.get('images', []):
            img_p = cj.parent / info['file_name']
            if not img_p.exists():
                continue
            w, h = int(info['width']), int(info['height'])
            mask = np.zeros((h, w), dtype=np.uint8)
            for t_cls in LAYER_ORDER:
                for a in anns_by_img.get(info['id'], []):
                    if cat_to_canon.get(a['category_id'], 0) != t_cls:
                        continue
                    seg = a.get('segmentation')
                    if isinstance(seg, dict):
                        rle = seg.copy()
                        if isinstance(rle.get('counts'), str):
                            rle['counts'] = rle['counts'].encode('utf-8')
                        elif isinstance(rle.get('counts'), list):
                            rle = mask_utils.frPyObjects(rle, h, w)
                        mb = mask_utils.decode(rle)
                        if mb.ndim == 3:
                            mb = mb.any(axis=2)
                        if mb.shape != (h, w):
                            mb = cv2.resize(mb.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                        mask[mb.astype(bool)] = t_cls
                    elif isinstance(seg, list):
                        for poly in seg:
                            if len(poly) >= 6:
                                pts = np.round(np.array(poly, dtype=np.float32).reshape(-1, 2)).astype(np.int32)
                                cv2.fillPoly(mask, [pts], color=t_cls)
            out_m = mask_dir / f'{img_p.stem}.png'
            Image.fromarray(mask).save(out_m)
            fname_low = img_p.name.lower()
            dom = 'kids_ethnic' if 'kids_ethnic' in fname_low else ('wedding_formals' if 'wedding_formals' in fname_low else default_domain)
            rows.append({
                'image_id': img_p.stem,
                'domain': dom,
                'image_path': str(img_p),
                'mask_path': str(out_m),
                'has_dupatta': bool((mask == 3).sum() > 300),
            })
    return rows

all_rows = []
for zp in sorted(FYP2_GOLD_DIR.glob('*.zip')):
    all_rows.extend(rasterize_coco_zip(zp, V3_GOLD_DIR / 'cache_fyp2', 'adult_pret'))
for zp in sorted(V3_GOLD_DIR.glob('*.zip')):
    all_rows.extend(rasterize_coco_zip(zp, V3_GOLD_DIR / 'cache_v3', 'wedding_formals'))

df = pd.DataFrame(all_rows).drop_duplicates(subset=['image_id']).reset_index(drop=True)
print('=== UNIVERSAL V3 GOLD DATASET ===')
print(df.groupby(['domain', 'has_dupatta']).size())

# Stratified split across (domain + dupatta)
df['strat_key'] = df['domain'] + '_' + df['has_dupatta'].astype(str)
valid_strat = df['strat_key'] if df['strat_key'].value_counts().min() >= 4 else df['domain']
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=valid_strat)
valid_strat2 = temp_df['domain'] if temp_df['domain'].value_counts().min() >= 2 else None
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=valid_strat2)
print(f'Split -> Train: {len(train_df)} | Val: {len(val_df)} | Unseen Test: {len(test_df)}')


In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes', do_reduce_labels=False)

class UniversalPKDataset(Dataset):
    def __init__(self, frame, size=(512, 512), augment=False):
        self.frame = frame.reset_index(drop=True)
        self.size = size
        self.augment = augment
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB').resize(self.size, Image.BILINEAR)
        msk = Image.open(row['mask_path']).resize(self.size, Image.NEAREST)
        if self.augment:
            if np.random.rand() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                msk = msk.transpose(Image.FLIP_LEFT_RIGHT)
            if np.random.rand() < 0.4:
                arr = np.clip(np.array(img).astype(np.float32) * np.random.uniform(0.88, 1.12), 0, 255).astype(np.uint8)
                img = Image.fromarray(arr)
        enc = processor(images=img, return_tensors='pt')
        return {'pixel_values': enc['pixel_values'].squeeze(0), 'labels': torch.tensor(np.array(msk), dtype=torch.long)}

pin = (device == 'cuda')
train_loader = DataLoader(UniversalPKDataset(train_df, augment=True), batch_size=4, shuffle=True, num_workers=2, pin_memory=pin)
val_loader = DataLoader(UniversalPKDataset(val_df), batch_size=4, shuffle=False, num_workers=2, pin_memory=pin)

model = SegformerForSemanticSegmentation.from_pretrained(
    'mattmdjaga/segformer_b2_clothes',
    num_labels=4,
    ignore_mismatched_sizes=True,
    id2label={0: 'other', 1: 'kurta_kameez', 2: 'bottom', 3: 'dupatta'},
    label2id={'other': 0, 'kurta_kameez': 1, 'bottom': 2, 'dupatta': 3},
).to(device)

class_weights = torch.tensor([0.5, 1.0, 2.0, 3.5], device=device)
ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

def dice_loss(logits, targets, eps=1e-6):
    probs = F.softmax(logits.float(), dim=1)
    one_hot = F.one_hot(targets, num_classes=4).permute(0, 3, 1, 2).float()
    inter = (probs * one_hot).sum(dim=(0, 2, 3))
    union = probs.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))
    return 1.0 - ((2.0 * inter + eps) / (union + eps)).mean()

@torch.no_grad()
def eval_iou(m, frame, is_baseline_18=False):
    m.eval()
    loader = DataLoader(UniversalPKDataset(frame), batch_size=4, shuffle=False, num_workers=2)
    inter, union = np.zeros(4, dtype=np.float64), np.zeros(4, dtype=np.float64)
    for batch in loader:
        pv = batch['pixel_values'].to(device)
        y = batch['labels'].to(device)
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            logits = m(pixel_values=pv).logits
        logits = F.interpolate(logits.float(), size=y.shape[-2:], mode='bilinear', align_corners=False)
        raw_p = logits.argmax(dim=1)
        if is_baseline_18:
            pred = torch.zeros_like(raw_p)
            pred[(raw_p == 5) | (raw_p == 6)] = 2
            pred[(raw_p == 4) | (raw_p == 7)] = 1
            pred[raw_p == 17] = 3
        else:
            pred = raw_p
        for c in range(4):
            inter[c] += ((pred == c) & (y == c)).sum().item()
            union[c] += ((pred == c) | (y == c)).sum().item()
    ious = {name: round(float(100.0 * inter[i] / max(union[i], 1.0)), 2) for i, name in enumerate(['other', 'kurta_kameez', 'bottom', 'dupatta'])}
    ious['mIoU'] = round(float(np.mean(list(ious.values()))), 2)
    return ious

optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
best_miou = -1.0
EPOCHS = 25

for ep in range(1, EPOCHS + 1):
    model.train()
    tot = 0.0
    for batch in tqdm(train_loader, desc=f'Epoch {ep:02d}/{EPOCHS}', leave=False):
        optimizer.zero_grad(set_to_none=True)
        pv, y = batch['pixel_values'].to(device, non_blocking=True), batch['labels'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            logits = F.interpolate(model(pixel_values=pv).logits, size=y.shape[-2:], mode='bilinear', align_corners=False)
            loss = ce_loss_fn(logits, y) + 0.5 * dice_loss(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tot += loss.item()
    v_iou = eval_iou(model, val_df)
    print(f'Epoch {ep:02d}/{EPOCHS} | loss={tot/max(len(train_loader),1):.4f} | val={v_iou}')
    if v_iou['mIoU'] > best_miou:
        best_miou = v_iou['mIoU']
        model.save_pretrained(RUN_DIR / 'best_checkpoint')
        processor.save_pretrained(RUN_DIR / 'best_checkpoint')

# Final Per-Domain Evaluation (All vs Adult vs Kids vs Wedding Formals) + Exact Baseline
best_m = SegformerForSemanticSegmentation.from_pretrained(RUN_DIR / 'best_checkpoint').to(device).eval()
base_m = SegformerForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes').to(device).eval()

report = {'overall_ours': eval_iou(best_m, test_df), 'overall_baseline': eval_iou(base_m, test_df, is_baseline_18=True), 'per_domain': {}}
for dom in sorted(test_df['domain'].unique()):
    sub = test_df[test_df['domain'] == dom]
    report['per_domain'][dom] = {
        'N': len(sub),
        'baseline': eval_iou(base_m, sub, is_baseline_18=True),
        'ours': eval_iou(best_m, sub),
    }

print('\n=== UNIVERSAL V3 PER-DOMAIN REPORT ===')
print(json.dumps(report, indent=2))
(PAPER_DIR / 'v3_universal_segformer_results.json').write_text(json.dumps(report, indent=2))
